Access the apiKey and documentation here: https://open.fda.gov/

In [33]:
import requests
import pandas as pd

In [34]:
# Define the base URL
base_url = "https://api.fda.gov/drug/ndc.json"

In [35]:
# Enter your API Key
api_key = "qf2TyQIxlujsbxBXPyMoiCYbAp2xtfkIYvvw1bAL"

In [36]:
# Specify your search parameters
searchable_fields = ["product_id",
                     "product_ndc",
                     "spl_id",
                     "product_type",
                     "finished",
                     "brand_name",
                    # "generic_name",
                     "dosage_form",
                    # "route",
                    "marketing_start_date",
                     "listing_expiration_date"]
                    # "marketing_category",
                    # "application_number",
                    # "pharm_class",
                   #  "dea_schedule",
                   #  ""] 

In [37]:
# Create an empty list to store the data
data = []

In [38]:
for field in searchable_fields:
    search_param = f"finished:true+AND+(brand_name:Simlandi+OR+brand_name:Humira+OR+brand_name:Cyltezo+OR+brand_name:Hyrimoz+OR+brand_name:Idacio+OR+brand_name:Amjevita+OR+brand_name:Hadlima)+AND+{field}:*"
    limit_param = 200

    query_url = f"{base_url}?api_key={api_key}&search={search_param}&limit={limit_param}"

    response = requests.get(query_url)
    
    if response.status_code == 200:
        results = response.json().get("results", [])
        for item in results:
            data.append(item)
    else:
        print(f"Error fetching data for field '{field}': {response.status_code} - {response.text}")

In [39]:
df = pd.DataFrame(data)

packaging_df = df.explode("packaging") # Each element of the packaging array becomes its own row


# Reset Index
packaging_df = packaging_df.reset_index(drop=True)

# Expand the dicts within the 'packaging' field into separate columns
packaging_details = pd.json_normalize(packaging_df["packaging"]).reset_index(drop=True)  # Reset index for normalized details


# Verify and Handle Column Overlaps
common_columns = packaging_df.columns.intersection(packaging_details.columns)
if not common_columns.empty:
    packaging_details = packaging_details.rename(columns=lambda col: f"{col}_details" if col in common_columns else col)


# Merge the flattened details back into the main DataFrame
final_df = pd.concat([packaging_df.drop(columns=["packaging"]), packaging_details], axis=1)


final_df.drop('openfda', axis=1, inplace=True)

In [41]:
# Save the DataFrame to a CSV file for storage
final_df.to_csv("drug_data.csv", index=False)

print("Data successfully retrieved and stored in 'drug_data.csv'")

Data successfully retrieved and stored in 'drug_data.csv'
